[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_14/2_sql.ipynb)

# Day 14: SQL (mock exam)

**Mock exam rules.** Set **one timer for 30 minutes** for the whole notebook (6 questions, about 5 minutes
each). Work as in a real exam: say the grain of each table and your plan out loud before typing. Do not open any
hint or solution until the timer rings. Then mark each question: solved in time, solved late, or not solved, and
put the last two in the mistake log.

Topics are mixed on purpose (CTEs, dates, ranking, top N, LAG / LEAD, running totals). Recognising which tool a
question needs is part of the test.

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (30 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | New or returning buyers? | medium | 5 |
| Q2 | Buyer quartiles | medium | 5 |
| Q3 | Favourite genre of each country | medium | 5 |
| Q4 | Time to the second order | medium | 5 |
| Q5 | Growth of the rater base | medium | 5 |
| Q6 | Peak day per platform, with context | medium | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: MovieLens small (GroupLens, research and education use).
def movielens(short):
    path = os.path.join(DATA, f"movielens_{short}.csv")
    if not os.path.exists(path):
        import io, zipfile
        print("downloading MovieLens")
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip").read()))
        for s in ("ratings", "movies", "tags", "links"):
            with open(os.path.join(DATA, f"movielens_{s}.csv"), "wb") as f:
                f.write(z.read(f"ml-latest-small/{s}.csv"))
    return pd.read_csv(path)

_r = movielens("ratings")
_r["rated_at"] = pd.to_datetime(_r["timestamp"], unit="s").dt.strftime("%Y-%m-%d %H:%M:%S")
_r.rename(columns={"userId": "user_id", "movieId": "movie_id"})[["user_id", "movie_id", "rating", "rated_at"]] \
    .to_sql("ml_ratings", db, index=False)
movielens("movies").rename(columns={"movieId": "movie_id"}).to_sql("ml_movies", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real tables: MovieLens small** (GroupLens; 610 users rated 9,742 movies, 1996 to 2018):

| Table | One row is | Columns |
|---|---|---|
| `ml_ratings` | one rating (100,836) | `user_id`, `movie_id`, `rating` (0.5 to 5), `rated_at` ('YYYY-MM-DD HH:MM:SS', UTC) |
| `ml_movies` | one movie (9,742) | `movie_id`, `title` (with the year), `genres` (several genres joined by a vertical bar, e.g. Comedy\|Romance) |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. New or returning buyers?

*medium, about 5 min*

Shopping app. Using **completed** orders only, for each month ('YYYY-MM') return `new_buyers`
(users whose first completed order is in that month), `returning_buyers` (users who bought this month and whose first
completed order was in an earlier month) and `buyers` (all distinct buyers that month).

Example: a user with completed orders on Jan 20, Feb 3 and Feb 18 counts as new in January and once as returning in
February.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: CTE with each user's first order date (a `MIN(...) OVER (PARTITION BY user_id)` or a separate group by), then conditional COUNT(DISTINCT CASE ...) per month.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: user_id, month of the order, `MIN(order_time) OVER (PARTITION BY user_id) AS first_order`.
2. Per month: `COUNT(DISTINCT CASE WHEN first month = month THEN user_id END)` and the same with `<`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH orders AS (
    SELECT user_id,
           substr(order_time, 1, 7) AS month,
           MIN(order_time) OVER (PARTITION BY user_id) AS first_order
    FROM purchases
    WHERE status = 'completed'
)
SELECT month,
       COUNT(DISTINCT CASE WHEN substr(first_order, 1, 7) = month THEN user_id END) AS new_buyers,
       COUNT(DISTINCT CASE WHEN substr(first_order, 1, 7) < month THEN user_id END) AS returning_buyers,
       COUNT(DISTINCT user_id) AS buyers
FROM orders
GROUP BY month
ORDER BY month
```

Result:

| month | new_buyers | returning_buyers | buyers |
|---|---|---|---|
| 2024-01 | 155 | 0 | 155 |
| 2024-02 | 185 | 65 | 250 |
| 2024-03 | 63 | 99 | 162 |

**Why:** The window MIN attaches the first order date to every order row without a join. COUNT(DISTINCT CASE ...) counts a user once per month even with several orders (CASE without ELSE gives NULL, which COUNT ignores). new + returning = buyers in every month, a quick sanity check. 'YYYY-MM' strings compare correctly as text because they are zero padded.

**Complexity:** One window sort over purchases plus one group by.

**Common mistakes:** COUNT without DISTINCT (counts orders). Computing the first order over all statuses while counting only completed ones (inconsistent definitions). Comparing full timestamps to months.

**Learn more:** [sql-cte](https://mahsa-agz.github.io/ds-handbook/#sql-cte), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates), [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when)

</details>

---
### Q2. Buyer quartiles

*medium, about 5 min*

Shopping app. Split all buyers (users with completed revenue) into **4 equal-size groups by completed
revenue**, quartile 1 = biggest spenders (ties: smaller user_id first). Per quartile return `buyers`, `min_rev`,
`max_rev`, `revenue` and `pct_revenue` (share of all completed revenue, 1 decimal).

Example: with 8 buyers, quartile 1 holds the 2 largest, quartile 4 the 2 smallest. If the count does not divide by 4,
the first groups get one extra buyer.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: NTILE(4) over the revenue order, then a group by on the bucket.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE spend per user.
2. `NTILE(4) OVER (ORDER BY revenue DESC, user_id)`.
3. Group by quartile; divide by the total from a scalar subquery.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH spend AS (
    SELECT user_id, SUM(amount) AS revenue
    FROM purchases
    WHERE status = 'completed'
    GROUP BY user_id
), q AS (
    SELECT user_id, revenue, NTILE(4) OVER (ORDER BY revenue DESC, user_id) AS quartile
    FROM spend
)
SELECT quartile,
       COUNT(*) AS buyers,
       ROUND(MIN(revenue), 2) AS min_rev,
       ROUND(MAX(revenue), 2) AS max_rev,
       ROUND(SUM(revenue), 2) AS revenue,
       ROUND(100.0 * SUM(revenue) / (SELECT SUM(revenue) FROM spend), 1) AS pct_revenue
FROM q
GROUP BY quartile
ORDER BY quartile
```

Result:

| quartile | buyers | min_rev | max_rev | revenue | pct_revenue |
|---|---|---|---|---|---|
| 1 | 101 | 129.45 | 620.32 | 20908.29 | 55.3 |
| 2 | 101 | 67.33 | 126.82 | 9322.18 | 24.7 |
| 3 | 101 | 37.47 | 66.55 | 5298.01 | 14.0 |
| 4 | 100 | 7.2 | 36.94 | 2270.08 | 6.0 |

**Why:** NTILE(n) deals rows into n buckets of (almost) equal size in window order; with 403 buyers the first 3 buckets get 101 and the last 100. The top quartile brings 55.3% of revenue. NTILE splits by row count, not by value: equal revenues can land in different buckets, so use PERCENT_RANK or fixed thresholds if the business wants value cutoffs.

**Complexity:** Group by plus one window sort over about 400 rows.

**Common mistakes:** Forgetting DESC (quartile 1 becomes the smallest spenders). Dividing by the quartile total instead of the grand total.

**Learn more:** [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
### Q3. Favourite genre of each country

*medium, about 5 min*

Chinook. For every customer country, return the genre with the **most units sold** (sum of
`InvoiceLine.Quantity`). If genres tie for first place, return all of them.

Return `country`, `genre`, `units`, ordered by country and genre. Use the customer's country.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: top 1 per group with ties: aggregate per (country, genre), RANK per country, keep rank 1.

</details>

<details><summary><b>Hint 2</b></summary>

1. Join Customer, Invoice, InvoiceLine, Track, Genre; group by country and genre.
2. `RANK() OVER (PARTITION BY country ORDER BY units DESC)`; outer filter `= 1`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH units AS (
    SELECT c.Country AS country, g.Name AS genre, SUM(il.Quantity) AS units
    FROM Customer c
    JOIN Invoice i      ON i.CustomerId = c.CustomerId
    JOIN InvoiceLine il ON il.InvoiceId = i.InvoiceId
    JOIN Track t        ON t.TrackId = il.TrackId
    JOIN Genre g        ON g.GenreId = t.GenreId
    GROUP BY c.Country, g.Name
), ranked AS (
    SELECT country, genre, units, RANK() OVER (PARTITION BY country ORDER BY units DESC) AS rk
    FROM units
)
SELECT country, genre, units
FROM ranked
WHERE rk = 1
ORDER BY country, genre
```

Result:

| country | genre | units |
|---|---|---|
| Argentina | Alternative & Punk | 9 |
| Argentina | Rock | 9 |
| Australia | Rock | 22 |
| Austria | Rock | 15 |
| Belgium | Rock | 21 |
| Brazil | Rock | 81 |
| Canada | Rock | 107 |
| Chile | Rock | 9 |
| Czech Republic | Rock | 25 |
| Denmark | Rock | 21 |
| Finland | Rock | 18 |
| France | Rock | 65 |

(25 rows, first 12 shown)

**Why:** RANK keeps ties, so Argentina returns two genres (Alternative & Punk and Rock, 9 units each). Rock wins almost everywhere; Sweden prefers Latin. A `GROUP BY country` with `MAX(units)` gives the number but not the genre, which is why the window is the standard tool.

**Complexity:** Joins over 2,240 invoice lines, a group by and a small window.

**Common mistakes:** ROW_NUMBER (drops one of the tied genres at random). Selecting `genre` next to `MAX(units)` in a plain GROUP BY (SQLite allows it and returns an arbitrary genre on ties; PostgreSQL rejects it).

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q4. Time to the second order

*medium, about 5 min*

Shopping app, all orders (any status). For each `platform`: `buyers` (users with at least one order),
`repeat_buyers` (users with a second order), `pct_repeat` (1 decimal) and `avg_days_to_2nd` (average days from the
first to the second order, 1 decimal, fractional days allowed).

Example: a user ordering on Jan 3 10:00 and Jan 5 22:00 has 2.5 days to the second order.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: LEAD (the next order of the same user) on the user's first row, found with ROW_NUMBER.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: ROW_NUMBER and `LEAD(order_time)` over `(PARTITION BY user_id ORDER BY order_time, order_id)`, join app_users for platform.
2. Keep n = 1; repeat = next_time IS NOT NULL; days with julianday.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH o AS (
    SELECT p.user_id, u.platform, p.order_time,
           ROW_NUMBER() OVER w        AS n,
           LEAD(p.order_time) OVER w  AS next_time
    FROM purchases p
    JOIN app_users u ON u.user_id = p.user_id
    WINDOW w AS (PARTITION BY p.user_id ORDER BY p.order_time, p.order_id)
)
SELECT platform,
       COUNT(*) AS buyers,
       SUM(CASE WHEN next_time IS NOT NULL THEN 1 ELSE 0 END) AS repeat_buyers,
       ROUND(100.0 * SUM(CASE WHEN next_time IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_repeat,
       ROUND(AVG(julianday(next_time) - julianday(order_time)), 1) AS avg_days_to_2nd
FROM o
WHERE n = 1
GROUP BY platform
ORDER BY platform
```

Result:

| platform | buyers | repeat_buyers | pct_repeat | avg_days_to_2nd |
|---|---|---|---|---|
| android | 205 | 114 | 55.6 | 10.4 |
| ios | 176 | 105 | 59.7 | 11.2 |
| web | 41 | 22 | 53.7 | 9.4 |

**Why:** On the first order row (n = 1), LEAD gives the second order time, or NULL for one-time buyers. AVG skips NULLs, so the average covers repeat buyers only, which is what we want. Repeat rates are close across platforms (about 54% to 60%). PostgreSQL: `EXTRACT(EPOCH FROM next_time - order_time) / 86400`.

**Complexity:** One window sort over purchases, then a group by.

**Common mistakes:** Taking `MAX - MIN` of order times (that is first to last, not first to second). Counting one-time buyers as 0 days in the average.

**Learn more:** [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q5. Growth of the rater base

*medium, about 5 min*

MovieLens (real). A user is a **new rater** in the year of their first rating. Return `yr`,
`new_raters` and `total_raters` (cumulative number of raters up to that year).

Example: 97 new raters in 1996 and 27 in 1997 give totals of 97 and 124.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: first event per user (MIN), count per year, then a running SUM over years.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: `MIN(substr(rated_at, 1, 4))` per user.
2. Group by year; `SUM(COUNT(*)) OVER (ORDER BY yr ROWS UNBOUNDED PRECEDING)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH firsts AS (
    SELECT user_id, MIN(substr(rated_at, 1, 4)) AS yr
    FROM ml_ratings
    GROUP BY user_id
)
SELECT yr,
       COUNT(*) AS new_raters,
       SUM(COUNT(*)) OVER (ORDER BY yr ROWS UNBOUNDED PRECEDING) AS total_raters
FROM firsts
GROUP BY yr
ORDER BY yr
```

Result:

| yr | new_raters | total_raters |
|---|---|---|
| 1996 | 97 | 97 |
| 1997 | 27 | 124 |
| 1998 | 9 | 133 |
| 1999 | 26 | 159 |
| 2000 | 48 | 207 |
| 2001 | 20 | 227 |
| 2002 | 14 | 241 |
| 2003 | 20 | 261 |
| 2004 | 11 | 272 |
| 2005 | 24 | 296 |
| 2006 | 23 | 319 |
| 2007 | 22 | 341 |

(23 rows, first 12 shown)

**Why:** Each user is counted once, in their first year, so the running total ends at 610 (all users). Counting distinct users per year and summing would count returning users many times.

**Complexity:** Group by over 100k ratings, then a window over 23 rows.

**Common mistakes:** `COUNT(DISTINCT user_id)` per year of any rating (active raters, not new raters). Forgetting the ORDER BY in the window (every row shows the grand total).

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-aggregation](https://mahsa-agz.github.io/ds-handbook/#sql-aggregation)

</details>

---
### Q6. Peak day per platform, with context

*medium, about 5 min*

Shopping app. For each `platform`, find the **day with the highest DAU** (users of that platform in
`daily_activity`; ties: earliest day) and show the platform's **7-day average DAU** ending on that day.

Return `platform`, `peak_day`, `peak_dau`, `dau_7d` (1 decimal).

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: daily counts per platform, a moving average per platform, then top 1 per platform. The moving average must be computed before you keep only the peak row.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE dau: platform, day, COUNT(*).
2. CTE: `AVG(dau) OVER (PARTITION BY platform ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)`.
3. CTE: `ROW_NUMBER() OVER (PARTITION BY platform ORDER BY dau DESC, day)`; keep 1.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH dau AS (
    SELECT u.platform, d.activity_date, COUNT(*) AS dau
    FROM daily_activity d
    JOIN app_users u ON u.user_id = d.user_id
    GROUP BY u.platform, d.activity_date
), smooth AS (
    SELECT platform, activity_date, dau,
           AVG(dau) OVER (PARTITION BY platform ORDER BY activity_date
                          ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS dau_7d
    FROM dau
), ranked AS (
    SELECT platform, activity_date, dau, dau_7d,
           ROW_NUMBER() OVER (PARTITION BY platform ORDER BY dau DESC, activity_date) AS rn
    FROM smooth
)
SELECT platform, activity_date AS peak_day, dau AS peak_dau, ROUND(dau_7d, 1) AS dau_7d
FROM ranked
WHERE rn = 1
ORDER BY platform
```

Result:

| platform | peak_day | peak_dau | dau_7d |
|---|---|---|---|
| android | 2024-02-24 | 147 | 135.4 |
| ios | 2024-02-19 | 127 | 118.3 |
| web | 2024-02-05 | 40 | 30.6 |

**Why:** Three steps, each one CTE: count, smooth, rank. The order matters: if you filtered to the peak row first, the moving average would only see one row. The peak is clearly above the 7-day average on every platform (for example android 147 versus 135.4), so a single day is a noisy summary; dashboards should show the smoothed line next to the raw one.

**Complexity:** Group by, two window sorts per platform over 91 days.

**Common mistakes:** Partitioning the moving average by day instead of platform. Using MAX(dau) in a group by and losing the date. Ranking before smoothing.

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-cte](https://mahsa-agz.github.io/ds-handbook/#sql-cte)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_14/3_stats.ipynb)